<a href="https://colab.research.google.com/github/intisariapps-com/Intisari-AutoCut-Android/blob/main/AutoCut_Studio_WebUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🎬 AutoCut Video Studio — Web Studio Visual Mandiri (Next.js Edition)

> 🌟 **Antarmuka Visual Next.js Lengkap:** Menghadirkan Web Studio modern untuk kurasi momen viral otomatis, preview template hook, rasio vertikal 9:16 reframe, dan galeri klip.
> 🧠 **100% Bebas API Key (Model AI Lokal):** Analisis transkrip YouTube langsung dieksekusi oleh model AI lokal resmi Qwen2.5 di GPU NVIDIA T4 Colab tanpa kuota/ketergantungan eksternal.
> 🌐 **Public HTTPS Cloudflare CDN Tunnel:** Menghasilkan tautan publik resmi yang kebal blokir ISP di Indonesia dengan kecepatan stream/download maksimal (> 10 MB/s).
> ⚡ **Zero-Configuration Connection (Single-Origin):** Antarmuka web otomatis terhubung langsung ke backend GPU Colab pada domain yang sama tanpa kendala CORS.
> 🛡️ **Protected Zero-Pip Engine:** Memuat aset web terkompilasi dan biner terenkripsi PyArmor `autocut_video_engine.zip`.

---
### 🚀 Cara Menjalankan:
1. Pastikan runtime GPU aktif (**Runtime** → **Change runtime type** → **T4 GPU**).
2. Klik tombol **Play (▶)** pada sel kode di bawah ini.
3. Tunggu ~30–40 detik hingga sistem siap. **Klik tautan resmi trycloudflare.com yang tercetak di layar untuk membuka AutoCut Web Studio di Tab Browser Baru (Layar Penuh)**.


In [ ]:
"""
🎬 AUTOCUT VIDEO STUDIO — WEB STUDIO VISUAL NEXT.JS (V1.8.0)
Hak Cipta (C) 2026 IntisariApps.com. Seluruh hak cipta dilindungi.
Menyajikan Antarmuka Visual Next.js + Backend Kurasi AI Qwen2.5 & Render GPU Colab.
"""

# @title ⚙️ PUSAT KENDALI ENGINE RENDERING COLAB
# @markdown Atur parameter sesi Colab di bawah ini:
AUTO_SHUTDOWN_MINUTES = 10  # @param [0, 5, 10, 15, 30] {type:"raw"}
GOOGLE_DRIVE_SYNC = "oauth_persistent"  # @param ["oauth_broker", "oauth_persistent", "native_mount", "oauth_api", "disabled"]
DRIVE_OAUTH_BROKER_URL = "https://autocut-drive-oauth.ripadienterfener.workers.dev"  # @param {type:"string"}
AUTOCUT_DRIVE_BROKER_TOKEN = ""  # @param {type:"string"}
GDRIVE_MASTER_FOLDER_ID = ""  # @param {type:"string"}
COOKIE_SOURCE = "gdrive"  # @param ["gdrive", "legacy", "disabled"]
GROQ_API_KEY = ""  # @param {type:"string"}
HF_TOKEN = ""  # @param {type:"string"}

import os
import sys
import time
import json
import subprocess
import zipfile
import sysconfig
import urllib.request

os.environ["GDRIVE_FOLDER_NAME"] = "AutoCut_Studio/Clips"
os.environ["GDRIVE_FOLDER_NAME"] = "AutoCut_Studio/Clips"
if "COOKIE_SOURCE" in globals():
    os.environ["COOKIE_SOURCE"] = str(COOKIE_SOURCE).lower().strip()

# Simpan API Key & Kredensial ke environment jika diisi
if "GROQ_API_KEY" in globals() and GROQ_API_KEY and GROQ_API_KEY.strip():
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY.strip()
if "HF_TOKEN" in globals() and HF_TOKEN and HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()

# Muat kredensial R2 dari Colab Secrets jika tersedia
try:
    from google.colab import userdata
    for secret_key in ["R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_BUCKET_NAME", "R2_PUBLIC_DOMAIN", "R2_ENDPOINT_URL"]:
        val = userdata.get(secret_key)
        if val:
            os.environ[secret_key] = str(val).strip()
except Exception:
    pass

# Drive authentication: persistent API never mounts or opens interactive auth.
os.environ["GOOGLE_DRIVE_SYNC"] = GOOGLE_DRIVE_SYNC
os.environ["COOKIE_DRIVE_MODE"] = "oauth_persistent" if GOOGLE_DRIVE_SYNC == "oauth_persistent" else "native_mount"
os.environ.pop("GDRIVE_MOUNT_PATH", None)
os.environ.pop("GDRIVE_USE_AUTH_USER", None)
os.environ.pop("GDRIVE_TOKEN_JSON", None)
os.environ["GDRIVE_MASTER_FOLDER_ID"] = GDRIVE_MASTER_FOLDER_ID.strip()
if GOOGLE_DRIVE_SYNC == "oauth_persistent":
    try:
        from google.colab import userdata
        token_json = userdata.get("GDRIVE_TOKEN_JSON")
        token_data = json.loads(token_json)
        if not all(token_data.get(k) for k in ("refresh_token", "client_id", "client_secret")):
            raise ValueError("missing_fields")
        os.environ["GDRIVE_TOKEN_JSON"] = token_json
        print("[Google Drive] Kredensial dimuat dari Secrets. Validasi API dilakukan engine; tanpa mount.")
    except Exception:
        raise RuntimeError("Tambahkan GDRIVE_TOKEN_JSON ke Colab Secrets dan aktifkan Notebook access. Buat token dengan generate_gdrive_token.py --read-existing; atau pilih native_mount.") from None
elif GOOGLE_DRIVE_SYNC == "oauth_broker":
    from google.colab import userdata
    broker_url = (DRIVE_OAUTH_BROKER_URL.strip() or "https://autocut-drive-oauth.ripadienterfener.workers.dev").rstrip("/")
    if not broker_url.startswith("https://"):
        raise RuntimeError("Isi DRIVE_OAUTH_BROKER_URL dengan origin HTTPS layanan OAuth.")
    try:
        control_key = (userdata.get("AUTOCUT_DRIVE_CONTROL_KEY") or "").strip()
        if len(control_key) < 32:
            raise ValueError("short_key")
    except Exception:
        import secrets
        control_key = secrets.token_urlsafe(32)
        print(f"[Google Drive] 🔑 AUTOCUT_DRIVE_CONTROL_KEY otomatis dibuat: {control_key}")
    os.environ["DRIVE_OAUTH_BROKER_URL"] = broker_url
    os.environ["AUTOCUT_DRIVE_CONTROL_KEY"] = control_key
    broker_token = ""
    if "AUTOCUT_DRIVE_BROKER_TOKEN" in globals() and AUTOCUT_DRIVE_BROKER_TOKEN and AUTOCUT_DRIVE_BROKER_TOKEN.strip():
        broker_token = AUTOCUT_DRIVE_BROKER_TOKEN.strip()
    else:
        try:
            broker_token = (userdata.get("AUTOCUT_DRIVE_BROKER_TOKEN") or "").strip()
        except Exception:
            pass
    if broker_token:
        os.environ["AUTOCUT_DRIVE_BROKER_TOKEN"] = broker_token
        os.environ["COOKIE_DRIVE_MODE"] = "oauth_broker"
        print("[Google Drive] 🔑 AUTOCUT_DRIVE_BROKER_TOKEN aktif; sinkronisasi klip dan cookies Google Drive otomatis.")
    else:
        if os.environ.get("COOKIE_SOURCE") == "gdrive":
            os.environ["COOKIE_SOURCE"] = "legacy"
            print("[Google Drive] Mode broker tanpa broker token; cookie memakai sumber legacy/lokal.")
        print("[Google Drive] Buka panel Drive pada WebUI untuk menghubungkan akun Anda.")
elif GOOGLE_DRIVE_SYNC == "native_mount":
    from google.colab import drive
    drive.mount("/content/drive")
    studio_root = "/content/drive/MyDrive/AutoCut_Studio"
    for subfolder in ("Cookies/youtube", "Cookies/gemini", "Clips"):
        os.makedirs(os.path.join(studio_root, subfolder), exist_ok=True)
    os.environ["GDRIVE_MOUNT_PATH"] = os.path.join(studio_root, "Clips")
    print("[Google Drive] Native mount aktif.")
elif GOOGLE_DRIVE_SYNC == "oauth_api":
    from google.colab import auth
    auth.authenticate_user()
    os.environ["GDRIVE_USE_AUTH_USER"] = "1"
    if os.environ.get("COOKIE_SOURCE") == "gdrive":
        raise RuntimeError("Pembacaan cookie memerlukan oauth_persistent atau native_mount.")
else:
    os.environ["COOKIE_SOURCE"] = "disabled"
    print("[Google Drive] Integrasi dinonaktifkan.")

print("=" * 80)
# Dedicated Video Gateway R2 (video.intisariapps.com) aktif secara otomatis (Zero-Secret).
print("🎬 MEMULAI AUTOCUT VIDEO STUDIO (NEXT.JS & AI LOCAL GGUF)")
print("=" * 80)

# 0. Pastikan binary cloudflared tersedia untuk terowongan fallback instan
if not os.path.exists("/usr/local/bin/cloudflared"):
    try:
        subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=False)
        subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=False)
    except Exception:
        pass

# 1. Instal dependensi sistem dasar (Gradio, FastAPI, MediaPipe, llama-cpp-python)
# 1. Instal & verifikasi ImageMagick Hook Engine
subprocess.run(["apt-get", "install", "-y", "-qq", "imagemagick", "fonts-noto-color-emoji"], check=False)
import shutil
im_bin = shutil.which("magick") or shutil.which("convert")
if not im_bin:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "imagemagick", "fonts-noto-color-emoji"], check=False)
    im_bin = shutil.which("magick") or shutil.which("convert")
if im_bin:
    im_test = subprocess.run([im_bin, "-version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if "ImageMagick" in im_test.stdout:
        print(f"✨ [1/4] ImageMagick Engine aktif & terverifikasi: {im_bin}", flush=True)
print("📦 [1/4] Memeriksa & memasang dependensi (FastAPI, Uvicorn, MediaPipe, ImageMagick, yt-dlp, curl_cffi, gemini_webapi)...", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "fastapi", "uvicorn[standard]", "python-multipart", "mediapipe", "requests", "qrcode", "pydantic", "yt-dlp", "curl_cffi", "gemini_webapi"], check=False)

# 🦕 [1/4] Pastikan Deno & Node.js JS Runtime terpasang & terdeteksi untuk solver n-sig yt-dlp
if not shutil.which("deno") and not os.path.exists("/usr/local/bin/deno"):
    try:
        print("🦕 [1/4] Memasang Deno JS Engine terbaru untuk yt-dlp solver...", flush=True)
        subprocess.run("curl -fsSL https://deno.land/install.sh | sh", shell=True, check=False)
        user_deno = os.path.expanduser("~/.deno/bin/deno")
        if os.path.exists(user_deno):
            subprocess.run(["cp", "-f", user_deno, "/usr/local/bin/deno"], check=False)
            subprocess.run(["chmod", "+x", "/usr/local/bin/deno"], check=False)
    except Exception as e_deno:
        print(f"ℹ️ Deno install note: {e_deno}", flush=True)

deno_dir = os.path.expanduser("~/.deno/bin")
current_path = os.environ.get("PATH", "")
if deno_dir not in current_path:
    os.environ["PATH"] = f"{deno_dir}:/usr/local/bin:{current_path}"

active_deno = shutil.which("deno") or ("/usr/local/bin/deno" if os.path.exists("/usr/local/bin/deno") else None)
active_node = shutil.which("node") or ("/usr/bin/node" if os.path.exists("/usr/bin/node") else None)
print(f"✨ [1/4] JavaScript Runtime terdeteksi -> Deno: {active_deno or 'tidak aktif'} | Node.js: {active_node or 'tidak aktif'}", flush=True)



has_gpu = False
try:
    has_gpu = subprocess.run(["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
except Exception:
    has_gpu = False

if has_gpu:
    print("⚡ [1/4] Akselerator GPU T4 aktif! Memasang llama-cpp-python CUDA...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "llama-cpp-python", "--prefer-binary", "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu122"], check=False)
else:
    print("💻 [1/4] Mode CPU terdeteksi. Memeriksa binary wheel llama-cpp-python (anti-hang)...", flush=True)
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:", "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cpu", "llama-cpp-python"], check=False, timeout=35)
    except Exception:
        print("ℹ️ [1/4] Binary wheel CPU dilewati, menggunakan fallback AI Cloud.", flush=True)

# 2. Pra-unduh model AI Lokal resmi Qwen2.5 GGUF ke disk NVMe Colab (Eager Warm-Up)
model_cache_dir = "/content/models"
model_target_file = os.path.join(model_cache_dir, "qwen2.5-1.5b-instruct-q4_k_m.gguf")
if not os.path.exists(model_target_file) or os.path.getsize(model_target_file) < 900 * 1024 * 1024:
    os.makedirs(model_cache_dir, exist_ok=True)
    print("🧠 [2/4] Mengunduh model AI resmi Qwen2.5-1.5B GGUF (~1.1 GB)...", flush=True)
    gguf_dl_url = "https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/qwen2.5-1.5b-instruct-q4_k_m.gguf"
    try:
        urllib.request.urlretrieve(gguf_dl_url, model_target_file + ".part")
        os.replace(model_target_file + ".part", model_target_file)
        print("✅ Model AI Qwen2.5 siap di disk Colab!", flush=True)
    except Exception as e_gguf_dl:
        print(f"⚠️ Pra-unduh model AI: {e_gguf_dl}", flush=True)
else:
    print("✅ [2/4] Model AI Qwen2.5 sudah tersedia di disk NVMe Colab.", flush=True)

# 3. Unduh & pasang modul engine terenkripsi resmi + Next.js Web Studio
print("🔐 [3/4] Mengunduh modul engine & Web Studio: autocut_video_engine.zip...")
pkg_url = f"https://raw.githubusercontent.com/intisariapps-com/Intisari-AutoCut-Android/main/autocut_video_engine.zip?t={int(time.time())}"
pkg_local = "/tmp/autocut_video_engine.zip"
try:
    subprocess.run(["wget", "-q", "--no-cache", "--no-cookies", "-O", pkg_local, pkg_url], check=True)
except Exception:
    urllib.request.urlretrieve(pkg_url, pkg_local)

pkg_size_kb = round(os.path.getsize(pkg_local) / 1024, 1) if os.path.exists(pkg_local) else 0.0
print(f"📦 Paket Biner Terpasang: autocut_video_engine.zip ({pkg_size_kb} KB)")
site_pkg = sysconfig.get_paths()["purelib"]
with zipfile.ZipFile(pkg_local, "r") as zf:
    zf.extractall(site_pkg)

# Bersihkan cache registri modul Python
for mod in list(sys.modules.keys()):
    if "autocut_video_engine" in mod or "pyarmor" in mod:
        del sys.modules[mod]

import autocut_video_engine
from autocut_video_engine.server import run_studio_server

# Inisialisasi folder AutoCut_Studio & subfolder Cookies/Clips di Google Drive
try:
    from autocut_video_engine.cookie_sources import bootstrap_gdrive_cookie_folder
    bootstrap_gdrive_cookie_folder()
except Exception as e_bs:
    pass

engine_ver = getattr(autocut_video_engine, "__version__", "1.8.0")
print(f"🚀 [4/4] Modul Terverifikasi: AutoCut Video Studio Engine v{engine_ver} (Next.js SPA Embedded)")

# 4. Jalankan Web Studio & Server GPU Colab via Single-Tunnel Cloudflare
run_studio_server(
    auto_shutdown_minutes=AUTO_SHUTDOWN_MINUTES,
    mode="webui",
    wait_forever=True
)